In [1]:
numCores = 8
print(f"Number of cores: {numCores}")

Number of cores: 8


In [2]:
import os
from glob import glob

# Topology file directory
topo_dir = "Data/Keggoro_abcd"#"pGRiNS/TOPOS"#
# Directory to store simulation results
sim_save_dir = "Data/SimulResults"
# Create output directory if it does not exist
#os.makedirs(sim_save_dir, exist_ok=True)

In [69]:
test = pd.read_csv("Data/Keggoro_abcd/Keggoro_abcd.topo",sep=" ")

In [61]:
import pandas as pd
a = pd.read_csv("Data/Keggoro_abcd/Keggoro_abcd2.topo",sep=" ")
b = pd.read_csv("Data/Topos/dorothea_abcd.topo",sep=" ")

In [57]:
a.drop_duplicates(subset=["Source","Target"])

,Source,Target,Type
0,A2M,PLAU,2
1,ABCA1,APOA1,1
2,ABL1,CTNNB1,2
3,ABL1,ENAH,1
4,ABL1,GRB2,1
...,...,...,...
29721,ZZZ3,WWC3,1
29722,ZZZ3,ZBTB20,1
29723,ZZZ3,ZMYM3,1
29724,ZZZ3,ZNF827,1


In [4]:
topo_files = [sorted(glob(f"{topo_dir}/*.topo"))[1]]
print(f"Number of topology files: {len(topo_files)}")
print(topo_files)

Number of topology files: 1
['Data/Keggoro_abcd/Keggoro_abcd.topo']


In [21]:
import sys
sys.path.append("./pGRiNS")

# RACIPE:

In [8]:
num_replicates = 1
num_params = 10000
num_init_conds = 100
sampling_method = "Uniform"
print(f"Number of replicates: {num_replicates}")
print(f"Number of parameters: {num_params}")
print(f"Number of initial conditions: {num_init_conds}\n")

Number of replicates: 1
Number of parameters: 10000
Number of initial conditions: 100



In [9]:
import jax.numpy as jnp
import grins.racipe_run as racipe
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
# Start multiprocessing pool
pool = mp.Pool(numCores)
print("Generating Parameter and Initial Condition files...")

# Parallel execution of file generation
pool.starmap(
    racipe.gen_topo_param_files,
    [
        (
            topo_file,
            sim_save_dir,
            num_replicates,
            num_params,
            num_init_conds,
            sampling_method
        )
        for topo_file in topo_files
    ],
)
print("Parameter and Initial Condition files generated.\n")

# Close multiprocessing pool
pool.close()

Generating Parameter and Initial Condition files...
Parameter and Intial Condition files generated for Keggoro_abcd
Parameter and Initial Condition files generated.



In [10]:

from importlib import reload
reload(racipe)
for topo_file in topo_files:
    # Generate parameters using Sobol sampling (optional - if the paramters are not already generated in parallel)
    """
    racipe.gen_topo_param_files(
        topo_file,
        sim_save_dir,
        num_replicates,
        num_params,
        num_init_conds,
        sampling_method="Uniform",
    )
    """
    
    
    racipe.run_all_replicates(
        topo_file,
        sim_save_dir,
        #tsteps=jnp.array([25.0, 75.0, 100.0]), # Run time-series simulations
        max_steps=1024, # originally 2048
        batch_size=6000,
    )


Loading ODE system from: Data/SimulResults/Keggoro_abcd
Number of combinations to simulate: 1000000
Running steady state simulations for replicate: 001
Time taken for replicate 001: 20.608466148376465
Normalising and Discretising the solutions


Array([[   0,    0],
       [   0,    1],
       [   0,    2],
       ...,
       [  99, 9997],
       [  99, 9998],
       [  99, 9999]], dtype=int32)

,VPS51,VTI1A,WIPF1,WWC3,ZBTB20,ZMYM3,ZNF462,ZNF827,ZP3,ZZZ3,...,gk_VTI1A,gk_WIPF1,gk_WWC3,gk_ZBTB20,gk_ZMYM3,gk_ZNF462,gk_ZNF827,gk_ZP3,gk_ZZZ3,State
0,55.472698,72.034897,17.707399,13.393300,16.603699,55.154999,72.110695,117.122696,349.957397,202.707001,...,0.867694,0.873501,0.996501,0.972931,0.583126,0.856079,0.784666,0.963000,1.000000,'0000000000'
1,5.285900,2.164000,1.153600,2.625000,124.909698,1.609700,0.525800,83.045197,1.920700,5.100200,...,0.035125,0.014400,0.027127,0.741707,0.016646,0.017144,1.000000,0.016671,0.999990,'0000000000'
2,46.986797,86.867195,100.899300,88.844894,83.472595,2.012600,17.321800,1.259500,68.117599,75.158501,...,0.652856,0.705392,0.780763,0.724781,0.027744,0.166304,0.040704,0.148735,1.000002,'0000000000'
3,16.186800,14.935800,103.423897,0.556100,18.267000,0.800600,121.867699,9.002299,9.797400,47.959400,...,0.147326,0.980228,0.044217,0.280289,0.024478,0.999866,0.040882,0.074627,1.000000,'0000000000'
4,0.038300,21.928499,6.902900,3.882100,2.697700,29.107800,0.227400,83.336395,1.042400,4.829500,...,0.193627,0.063532,0.388728,0.033175,0.100680,0.011093,0.128503,0.018303,1.000007,'0000000000'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,9.676700,346.433105,85.110100,81.325798,376.874390,44.920700,7.261100,32.626400,43.961899,96.848396,...,0.480934,0.812636,0.998860,0.734037,0.957800,0.160384,0.941611,0.359223,1.000000,'0000000000'
999996,85.890594,60.751900,111.674294,227.011993,113.203697,221.117798,172.356598,34.572498,52.139099,239.024200,...,0.999999,0.995675,0.989870,0.717786,0.937789,0.986447,0.783841,0.995539,1.000000,'0000000000'
999997,78.782700,249.380692,63.732800,50.541298,376.133789,206.255890,8.383400,66.860802,63.826797,119.879700,...,0.999856,0.844014,0.443075,0.999297,0.998305,0.217237,0.478608,0.999966,1.000000,'0000000000'
999998,0.722500,2.857700,1.487300,33.033501,314.658905,0.987300,0.584600,3.026700,0.487700,10.415000,...,0.049374,0.015615,0.102940,0.952714,0.028524,0.021047,0.028916,0.024624,0.999998,'0000000000'


Simulation completed for replicate: 001



# Boolise:

In [18]:
testoutrac = pd.read_parquet("Data/SimulResults/Keggoro_abcd/001/Keggoro_abcd_steadystate_solutions_001.parquet")

In [ ]:
testoutrac["InitCondNum"]

,Step,Initnum,VPS51|VTI1A|WIPF1|WWC3|ZBTB20|ZMYM3|ZNF462|ZNF827,ZP3|ZZZ3
0,0,0,118,32
1,100,0,127,224
2,0,1,137,192
3,100,1,137,192
4,0,2,176,192
...,...,...,...,...
32763,100,16381,255,224
32764,0,16382,179,224
32765,100,16382,255,224
32766,0,16383,204,0


In [12]:
import pandas as pd

In [ ]:
import jax.numpy as jnp
import grins.ising_bool as ising_bool
replacement_values = jnp.array([-1, 1])
max_steps = 100
print(f"Number of steps: {max_steps}")
num_initial_conditions = 2**14
print(f"Number of initial conditions: {num_initial_conditions}")
batch_size = 2**10
num_replicates = 3
save_dir = "SimulResultsB"

Number of steps: 100
Number of initial conditions: 16384


In [7]:
from importlib import reload
reload(ising_bool)
for topo_file in topo_files:
    """
    ising_bool.run_all_replicates_ising(
        topo_file,
        num_initial_conditions=num_initial_conditions,
        batch_size=batch_size,
        save_dir=save_dir,
        mode="sync",
        packbits=True,
        num_replicates=num_replicates
    )
    """
    ising_bool.run_all_replicates_ising(
        topo_file,
        num_initial_conditions=num_initial_conditions,
        save_dir=save_dir,
        mode="sync",
        packbits=True,
    )

Topology: Data/Keggoro_abcd/Keggoro_abcd.topo
Running sync simulations for the network: Data/Keggoro_abcd/Keggoro_abcd.topo
Simulation time for sync mode: 0.55 seconds
Running sync simulations for the network: Data/Keggoro_abcd/Keggoro_abcd.topo
Simulation time for sync mode: 0.10 seconds
Running sync simulations for the network: Data/Keggoro_abcd/Keggoro_abcd.topo
Simulation time for sync mode: 0.10 seconds
